# 22f — the (mu, gamma) grid as ONE big contour plot

**Question (Anuar).** One figure: for every (mu, gamma) combination on the 13x10 grid, show the
(FWHM, sigma_fit) cloud of our **three estimators** as contours, and add the **real data only at its own
(true mu, true gamma) node** (no more "real cloud in every panel").

**Rendered purely from the companion run data** `data/processed/22e_clouds.npz` (the 40x40 binned
densities). **No process is re-run** — 22e did the Monte-Carlo; 22f only re-presents it.

**Deliverable.** `22f-mu-gamma-grid-contours.html` — open in a browser: mouse-wheel zoom per panel,
drag-zoom, pan, and legend toggles.

## How to read — how mu and gamma map to panels
- **Rows = mu (photons)**, increasing **downward**: 3, 4, 6, 8, 12, 17, 24, 34, 48, 68, 96, 136, 192.
- **Columns = gamma (MHz)**, increasing **to the right**: 3, 4, 5, 6.5, 8.5, 11, 14, 18, 24, 31.
- Every panel is labelled **`mu=.., gamma=..`** at its top, and the outer annotations spell out the row/column
  meaning, so "which mu / which gamma am I looking at" is never ambiguous.
- Inside a panel, the two axes are always **FWHM (MHz, x)** vs **sigma_fit (MHz, y)**, both fixed to
  [0, 70] x [0, 40] so all panels are directly comparable.
- **Contours** (three estimators): `ours: Lorentzian MLE` (dark blue), `ours: pseudo-Voigt` (teal),
  `Gregor: binned Voigt LSQ` (orange) — iso-density of the binned 2-D cloud.
- **Real data**: each of the 14 experiments is drawn **only in the single panel whose (mu, gamma) is
  closest to that experiment's true (mu_true, gamma_true)**, in its own colour (legend = identity).
  *Caveat:* gamma is pinned to the experiment's own truth, and the grid is coarse in mu, so a few
  experiments round to the same node: **(mu=68, gamma=8.5)** holds 1nW T60/T80/T100 and
  **(mu=96, gamma=14)** holds 3nW T40/T60.
- **Switching (works in the exported HTML).** Every layer is a *plotly legend group*: click any legend
  entry to show/hide that layer (`legend.groupclick='togglegroup'`), so a click never changes only one
  panel.

In [1]:
# ============================================================
# 22f — imports (light: no torch, no Monte-Carlo — everything comes from the npz)
# ============================================================
import os, sys, time
import numpy as np
from scipy.ndimage import gaussian_filter

import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = 'plotly_white'
pio.renderers.default = os.environ.get('PLOTLY_RENDERER', 'vscode')

# climb to the repo root (has src/ and data/)
for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')) and os.path.isdir(os.path.join(_p, 'data')):
        REPO_ROOT = os.path.abspath(_p); break
os.chdir(REPO_ROOT)
print('Imports OK | plotly', plotly.__version__, '| repo', REPO_ROOT)


Imports OK | plotly 7.1.0 | repo /home/pukky/.openclaw/workspace/qm-ml


In [2]:
# ============================================================
# 22f CONFIG
# ============================================================
NPZ      = 'data/processed/22e_clouds.npz'
HTML_OUT = 'notebooks/22_distribution_dynamics/22f-mu-gamma-grid-contours.html'

X_RANGE  = (0.0, 70.0)   # FWHM  (MHz)
Y_RANGE  = (0.0, 40.0)   # sigma_fit (MHz)

EST_NAMES  = {'lorentzian':   'ours: Lorentzian MLE',
              'pseudo_voigt': 'ours: pseudo-Voigt fit',
              'gregor':       'Gregor: binned Voigt LSQ'}
EST_COLORS = {'lorentzian':   '#1d3557',
              'pseudo_voigt': '#2a9d8f',
              'gregor':       '#e76f51'}

# distinct colour per experiment (real data now lives at one node each; colour = identity)
EXP_COLORS = ['#e6194B', '#3cb44b', '#f58231', '#911eb4', '#42d4f4', '#f032e6', '#469990',
              '#9A6324', '#800000', '#808000', '#000075', '#008080', '#8B4513', '#c71585']

SMOKE = os.environ.get('NB_SMOKE') == '1'
print('config loaded | SMOKE =', SMOKE)


config loaded | SMOKE = False


In [3]:
# ============================================================
# 22f — EXPERIMENTS (true values from Gregor's fits; identical to 22a/22b/22c/22e)
# ============================================================
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393, sigma_prop=2.576, lam=2.232, gamma_true=8.5, n_target=61, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans05.txt'),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372, sigma_prop=3.445, lam=2.122, gamma_true=8.5, n_target=358, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans10.txt'),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316, sigma_prop=4.141, lam=2.286, gamma_true=8.5, n_target=1138, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans20.txt'),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405, sigma_prop=7.198, lam=2.351, gamma_true=8.5, n_target=2428, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans40.txt'),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374, sigma_prop=9.851, lam=2.593, gamma_true=8.5, n_target=2424, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans60.txt'),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365, sigma_prop=12.627, lam=2.758, gamma_true=8.5, n_target=2487, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans80.txt'),
    dict(name='1nW Trans100', power='1nW', mu_true=70.817, sigma_prop=17.221, lam=2.636, gamma_true=8.5, n_target=2455, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans100.txt'),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204, sigma_prop=3.724, lam=2.186, gamma_true=14.1, n_target=252, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans05.txt'),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476, sigma_prop=5.639, lam=2.158, gamma_true=14.1, n_target=1572, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans10.txt'),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279, sigma_prop=8.319, lam=2.264, gamma_true=14.1, n_target=2171, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans20.txt'),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892, sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans40.txt'),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95, lam=2.741, gamma_true=14.1, n_target=2541, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans60.txt'),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans80.txt'),
    dict(name='3nW Trans100', power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans100.txt'),
]
print(len(EXPERIMENTS), 'experiments')


14 experiments


In [4]:
# ============================================================
# 22f — LOAD the 22e companion data (no recompute) + real-data loader + node mapping
# ============================================================
_d = np.load(NPZ, allow_pickle=True)
MU  = _d['mu_grid'].astype(float)
GA  = _d['gamma_grid'].astype(float)
EST = [str(e) for e in _d['estimators']]
_S  = _d['stats']

DENS = {}                                            # (est, i, j) -> uint16 40x40 histogram
def _dens_from_H(H, nx=40, ny=40):
    """Reproduce 22e's _dens_xy rendering from the saved histogram: smooth -> normalise -> uint8."""
    Z = gaussian_filter(H.astype(float).T, 1.1)
    Z = Z / max(Z.max(), 1e-12)
    xe = np.linspace(X_RANGE[0], X_RANGE[1], nx + 1)
    ye = np.linspace(Y_RANGE[0], Y_RANGE[1], ny + 1)
    return 0.5 * (xe[:-1] + xe[1:]), 0.5 * (ye[:-1] + ye[1:]), Z.astype(float)   # Z in [0,1]

# --- iso-contour extraction (marching squares via matplotlib; computation only, no figure shown/saved) ---
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

BAND_LEVELS = (0.15, 0.35, 0.55, 0.75)
BAND_ALPHAS = (0.08, 0.11, 0.14, 0.17)

def cloud_contours(xc, yc, Z01, levels=BAND_LEVELS):
    if Z01.max() <= 0:
        return []
    _f = Figure(); FigureCanvasAgg(_f)
    _ax = _f.add_subplot(111)
    cs = _ax.contour(xc, yc, Z01, levels=list(levels))
    return [(i, float(lev), [seg for seg in segs if seg.shape[0] >= 3])
            for i, (lev, segs) in enumerate(zip(cs.levels, cs.allsegs))]

def _rgba(hexcol, a):
    h = hexcol.lstrip('#'); return f'rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})'

for ei, ek in enumerate(EST):
    for i in range(len(MU)):
        for j in range(len(GA)):
            DENS[(ek, i, j)] = _d[f'dens_{ei}_{i}_{j}']

def _real_data(exp):
    d = np.genfromtxt(exp['data_file'])
    f = d[:, 0] * 1000.0; e = d[:, 1] * 1000.0
    ok = ~np.isnan(f) & ~np.isnan(e) & (f > 0)
    flt = ok & (e / f < 10.0)
    return f[flt], e[flt]

REAL = {e['name']: _real_data(e) for e in EXPERIMENTS}

# each experiment -> its nearest (mu, gamma) grid node  (real data drawn ONLY there)
def _nearest(vals, v): return int(np.argmin(np.abs(vals - v)))
EXP_NODE = {e['name']: (_nearest(MU, e['mu_true']), _nearest(GA, e['gamma_true'])) for e in EXPERIMENTS}

print(f'loaded {len(EST)} estimators x {len(MU)} mu x {len(GA)} gamma = {len(DENS)} nodes from {NPZ}')
print('experiment -> node:')
for e in EXPERIMENTS:
    i, j = EXP_NODE[e['name']]
    print(f'  {e["name"]:12s} mu_true={e["mu_true"]:7.1f} gamma_true={e["gamma_true"]:4} '
          f'-> (mu={MU[i]:g}, gamma={GA[j]:g})   n_real={REAL[e["name"]][0].size}')


loaded 3 estimators x 13 mu x 10 gamma = 390 nodes from data/processed/22e_clouds.npz
experiment -> node:
  1nW Trans05  mu_true=    9.4 gamma_true= 8.5 -> (mu=8, gamma=8.5)   n_real=61
  1nW Trans10  mu_true=   12.4 gamma_true= 8.5 -> (mu=12, gamma=8.5)   n_real=358
  1nW Trans20  mu_true=   17.3 gamma_true= 8.5 -> (mu=17, gamma=8.5)   n_real=1138
  1nW Trans40  mu_true=   38.4 gamma_true= 8.5 -> (mu=34, gamma=8.5)   n_real=2428
  1nW Trans60  mu_true=   61.4 gamma_true= 8.5 -> (mu=68, gamma=8.5)   n_real=2424
  1nW Trans80  mu_true=   79.4 gamma_true= 8.5 -> (mu=68, gamma=8.5)   n_real=2487
  1nW Trans100 mu_true=   70.8 gamma_true= 8.5 -> (mu=68, gamma=8.5)   n_real=2455
  3nW Trans05  mu_true=   13.2 gamma_true=14.1 -> (mu=12, gamma=14)   n_real=252
  3nW Trans10  mu_true=   24.5 gamma_true=14.1 -> (mu=24, gamma=14)   n_real=1572
  3nW Trans20  mu_true=   34.3 gamma_true=14.1 -> (mu=34, gamma=14)   n_real=2171
  3nW Trans40  mu_true=   84.9 gamma_true=14.1 -> (mu=96, gamma=14)   n_

In [5]:
# ============================================================
# 22f — ONE big figure: 13 (mu) x 10 (gamma) panels
#   layer = plotly legendgroup  ->  one legend click toggles that layer (all panels)
# ============================================================
SUBSET_MU = [i for i in range(len(MU)) if (not SMOKE) or (i % 4 == 0)]
SUBSET_GA = [j for j in range(len(GA)) if (not SMOKE) or (j % 4 == 0)]

t0 = time.time()
fig = make_subplots(
    rows=len(SUBSET_MU), cols=len(SUBSET_GA),
    horizontal_spacing=0.012, vertical_spacing=0.020,
    subplot_titles=[f'mu={MU[i]:g}, gamma={GA[j]:g}' for i in SUBSET_MU for j in SUBSET_GA],
)

seen = set()
def _first(grp):
    s = grp not in seen; seen.add(grp); return s

# ---- pass 1: ALL real-data points (bottom layer — contours draw OVER them) ----
for r, i in enumerate(SUBSET_MU, start=1):
    for c, j in enumerate(SUBSET_GA, start=1):
        for k, exp in enumerate(EXPERIMENTS):
            if EXP_NODE[exp['name']] != (i, j):
                continue
            rf, rs = REAL[exp['name']]
            fig.add_trace(go.Scatter(
                x=rf, y=rs, mode='markers',
                marker=dict(size=4, color=EXP_COLORS[k], opacity=0.9,
                            line=dict(width=0.4, color='rgba(20,20,20,0.55)')), hoverinfo='skip',
                legendgroup='real_' + exp['name'], name='real ' + exp['name'],
                showlegend=_first('real_' + exp['name']),
            ), r, c)

# ---- pass 2: ALL contour bands as SCATTER polygons (go.Contour is stuck in a layer BELOW scatter) ----
for r, i in enumerate(SUBSET_MU, start=1):
    for c, j in enumerate(SUBSET_GA, start=1):
        for ek in EST:
            xc, yc, Z = _dens_from_H(DENS[(ek, i, j)])
            for bi, lev, loops in cloud_contours(xc, yc, Z):
                for lp in loops:
                    fig.add_trace(go.Scatter(
                        x=lp[:, 0], y=lp[:, 1], mode='lines', fill='toself',
                        fillcolor=_rgba(EST_COLORS[ek], BAND_ALPHAS[bi]),
                        line=dict(color=EST_COLORS[ek], width=0.9),
                        hoverinfo='skip', legendgroup='est_' + ek, name=EST_NAMES[ek],
                        showlegend=_first('est_' + ek)), r, c)
        fig.update_xaxes(range=X_RANGE, showticklabels=(r == len(SUBSET_MU)), row=r, col=c)
        fig.update_yaxes(range=Y_RANGE, showticklabels=(c == 1), row=r, col=c)

fig.update_layout(
    height=175 * len(SUBSET_MU) + 130,
    width=195 * len(SUBSET_GA) + 130,
    legend=dict(groupclick='togglegroup', font=dict(size=11), itemsizing='constant'),
    margin=dict(l=90, r=20, t=80, b=45),
    title_text='22f — (FWHM, sigma_fit) contours over the (mu, gamma) grid  |  rows = mu (photons), '
               'cols = gamma (MHz)  |  real data at its own (mu, gamma) node  |  click legend to toggle',
)
fig.update_annotations(font=dict(size=10))
# row / column meaning
fig.add_annotation(text='<b>gamma (MHz) &rarr;</b>', xref='paper', yref='paper', x=0.5, y=1.028,
                   showarrow=False, font=dict(size=14))
fig.add_annotation(text='<b>mu (photons) &darr;</b>', xref='paper', yref='paper', x=-0.045, y=0.5,
                   textangle=-90, showarrow=False, font=dict(size=14))

CONFIG = {'scrollZoom': True, 'displaylogo': False, 'responsive': True}
print(f'built {len(fig.data)} traces in {time.time()-t0:.1f} s')


built 1680 traces in 7.0 s


In [6]:
# ============================================================
# 22f — show inline + write the standalone HTML (browser: wheel-zoom per panel, pan, legend toggles)
# ============================================================
os.makedirs(os.path.dirname(HTML_OUT), exist_ok=True)
fig.write_html(HTML_OUT, include_plotlyjs='cdn', config=CONFIG)
print('wrote', HTML_OUT, f'({os.path.getsize(HTML_OUT)/1e6:.1f} MB)')

fig.show(config=CONFIG)


wrote notebooks/22_distribution_dynamics/22f-mu-gamma-grid-contours.html (1.9 MB)


## Diagram 2 — only the nodes that carry real data

Same format as Diagram 1, but restricted to the real-data nodes: **2 columns (1nW / 3nW) x one row per
transparency**. Each panel is one experiment at its own (mu, gamma) node — 3-estimator contours + its real
points. Where several experiments share a node (mu=68, gamma=8.5 for 1nW T60/T80/T100; mu=96, gamma=14 for
3nW T40/T60), the contours repeat and only the real dots and the title differ.


In [7]:
# ============================================================
# 22f DIAGRAM 2 — only the nodes that carry real data
#   2 columns (1nW / 3nW) x one row per transparency
# ============================================================
POWERS = ['1nW', '3nW']
TRANS  = ['Trans05', 'Trans10', 'Trans20', 'Trans40', 'Trans60', 'Trans80', 'Trans100']
EXP_IDX = {e['name']: k for k, e in enumerate(EXPERIMENTS)}
if SMOKE:
    TRANS = TRANS[::3]

t0 = time.time()
fig2 = make_subplots(
    rows=len(TRANS), cols=len(POWERS),
    horizontal_spacing=0.07, vertical_spacing=0.045,
    subplot_titles=[f'{p} {t}   (mu={MU[EXP_NODE[p + " " + t][0]]:g}, gamma={GA[EXP_NODE[p + " " + t][1]]:g})'
                    for t in TRANS for p in POWERS],
)

seen2 = set()
def _first2(grp):
    s = grp not in seen2; seen2.add(grp); return s

# ---- pass 1: ALL real-data points (bottom layer — contours draw OVER them) ----
for r, t in enumerate(TRANS, start=1):
    for c, p in enumerate(POWERS, start=1):
        name = f'{p} {t}'
        k = EXP_IDX[name]
        rf, rs = REAL[name]
        fig2.add_trace(go.Scatter(
            x=rf, y=rs, mode='markers',
            marker=dict(size=4, color=EXP_COLORS[k], opacity=0.9,
                        line=dict(width=0.4, color='rgba(20,20,20,0.55)')), hoverinfo='skip',
            legendgroup='real_' + name, name='real ' + name, showlegend=_first2('real_' + name),
        ), r, c)

# ---- pass 2: ALL contour bands as SCATTER polygons (go.Contour is stuck in a layer BELOW scatter) ----
for r, t in enumerate(TRANS, start=1):
    for c, p in enumerate(POWERS, start=1):
        name = f'{p} {t}'
        i, j = EXP_NODE[name]
        for ek in EST:
            xc, yc, Z = _dens_from_H(DENS[(ek, i, j)])
            for bi, lev, loops in cloud_contours(xc, yc, Z):
                for lp in loops:
                    fig2.add_trace(go.Scatter(
                        x=lp[:, 0], y=lp[:, 1], mode='lines', fill='toself',
                        fillcolor=_rgba(EST_COLORS[ek], BAND_ALPHAS[bi]),
                        line=dict(color=EST_COLORS[ek], width=1.0),
                        hoverinfo='skip', legendgroup='est_' + ek, name=EST_NAMES[ek],
                        showlegend=_first2('est_' + ek)), r, c)
        fig2.update_xaxes(range=X_RANGE, showticklabels=(r == len(TRANS)), row=r, col=c)
        fig2.update_yaxes(range=Y_RANGE, showticklabels=(c == 1), row=r, col=c)

fig2.update_layout(
    height=340 * len(TRANS) + 120,
    width=640 * len(POWERS) + 160,
    legend=dict(groupclick='togglegroup', font=dict(size=11), itemsizing='constant'),
    margin=dict(l=80, r=20, t=80, b=45),
    title_text='22f (diagram 2) — (FWHM, sigma_fit) at the real (mu, gamma) nodes  |  columns = laser power, '
               'rows = transparency  |  contours = our 3 estimators, dots = the real data of that experiment',
)
fig2.update_annotations(font=dict(size=11))
fig2.add_annotation(text='<b>1nW&nbsp;&nbsp;(gamma=8.5)</b>', xref='paper', yref='paper', x=0.25, y=1.022,
                    showarrow=False, font=dict(size=14))
fig2.add_annotation(text='<b>3nW&nbsp;&nbsp;(gamma=14)</b>', xref='paper', yref='paper', x=0.75, y=1.022,
                    showarrow=False, font=dict(size=14))
print(f'built {len(fig2.data)} traces in {time.time()-t0:.1f} s')


built 198 traces in 0.7 s


In [8]:
# ============================================================
# 22f DIAGRAM 2 — write the standalone HTML
# ============================================================
HTML_OUT2 = 'notebooks/22_distribution_dynamics/22f-real-data-grid.html'
fig2.write_html(HTML_OUT2, include_plotlyjs='cdn', config=CONFIG)
print('wrote', HTML_OUT2, f'({os.path.getsize(HTML_OUT2)/1e6:.1f} MB)')

fig2.show(config=CONFIG)


wrote notebooks/22_distribution_dynamics/22f-real-data-grid.html (0.8 MB)


## Notes
- **Source of truth.** Every contour comes from `data/processed/22e_clouds.npz` (22e's 40x40 binned
  densities); nothing is re-simulated. Real data is the full valid point set per experiment (no decimation),
  since each experiment now occupies a single panel.
- **Panel = (mu, gamma).** Rows are mu (photons), columns are gamma (MHz), both increasing downward / to the
  right; every panel is labelled `mu=.., gamma=..`, and the two in-panel axes are always FWHM x sigma_fit
  over the fixed ranges [0,70] x [0,40].
- **Real data placement.** Each experiment is drawn only at the grid node nearest its true (mu, gamma).
  Because the grid is coarse in mu, a few experiments share a node: (mu=68, gamma=8.5) holds 1nW
  T60/T80/T100 and (mu=96, gamma=14) holds 3nW T40/T60 — use the legend to isolate them.
- **The switcher.** Layers are plotly legend groups with `groupclick='togglegroup'`; a single legend click
  flips that layer across the whole figure (native to the exported HTML).
- **No raw scatter for the simulated clouds** — only binned densities were persisted by 22e (contours, as
  requested). Companion artefacts: `22e_clouds.npz` (data), `22f-mu-gamma-grid-contours.html`
  (Diagram 1, full 13x10 grid) and `22f-real-data-grid.html` (Diagram 2, real-data nodes only).
